# Découpage spatial des données — Région Casablanca-Settat

## Projet SIG : Évaluation de la vulnérabilité aux inondations urbaines dans le Grand Casablanca

Ce notebook présente le découpage spatial des données géographiques
utilisées dans le cadre du projet, à partir de la limite administrative
de la région Casablanca-Settat.

### Objectifs
- Définir l'emprise géographique de la zone d'étude.
- Découper les couches géographiques OpenStreetMap (OSM).
- Découper les couches hydrographiques issues de CartoMaroc.
- Enregistrer les résultats dans une structure de dossiers organisée.


### Sources de données
- **Source 1 :** OpenStreetMap — données géographiques générales.
- **Source 2 :** CartoMaroc — données hydrographiques.

### Référentiel géographique
- **Zone d'étude :** Casablanca-Settat, Maroc.
- **CRS de la limite utilisée  / source:** EPSG:4326.
- **Format de sortie :** GeoPackage (`.gpkg`).
- 
Les données brutes sont conservées intactes. Les couches découpées
sont enregistrées dans le dossier `donnees_traitees/`.


## 1. Définition de l'emprise et découpage des données OSM

Cette étape consiste à identifier la limite administrative de
Casablanca-Settat, à vérifier sa géométrie et à l'utiliser pour
découper les couches OpenStreetMap.

Les couches obtenues sont enregistrées au format GeoPackage dans
`donnees_traitees/casablanca_settat/osm/`.

In [2]:

from pathlib import Path
import geopandas as gpd
import pandas as pd

# -----------------------------------------------------------------------------
# CONFIGURATION DES CHEMINS
# -----------------------------------------------------------------------------
# Le notebook est exécuté depuis Geodata/scripts
ROOT = Path.cwd().resolve().parent

# Données brutes OSM
OSM_DIR = ROOT / "donnees_brutes" / "osm"
ADMIN_PATH = OSM_DIR / "gis_osm_adminareas_a_free_1.shp"

# Données traitées : à l'extérieur de scripts
OUT_DIR = (
    ROOT / "donnees_traitees"
    / "casablanca_settat"
    / "osm"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Dossier racine :", ROOT)
print("Dossier OSM brut :", OSM_DIR.resolve())
print("Dossier OSM traité :", OUT_DIR.resolve())


# 1. Lire les limites administratives
admin = gpd.read_file(ADMIN_PATH)

# 2. Sélectionner la région Casablanca-Settat
limite = admin[
    (admin["osm_id"].astype(str) == "2424085")
    & (admin["fclass"] == "admin_level4")
].copy()

if limite.empty:
    raise ValueError(
        "Limite Casablanca-Settat introuvable : vérifie osm_id et fclass."
    )

if limite.crs is None:
    raise ValueError("Le CRS de la limite administrative est absent.")

# Réparer la géométrie si nécessaire
limite["geometry"] = limite.geometry.make_valid()
limite = limite[
    limite.geometry.notna() & ~limite.geometry.is_empty
]
limite = limite.dissolve()

# Sauvegarder la limite utilisée
limite.to_file(
    OUT_DIR.parent / "limite_casablanca_settat.gpkg",
    driver="GPKG"
)

print("Limite administrative chargée.")
print("CRS de la limite :", limite.crs)
print("Dossier de sortie :", OUT_DIR)

# 3. Découper chaque shapefile OSM
shapefiles = [
    p for p in OSM_DIR.glob("*.shp")
    if p.name.lower() != ADMIN_PATH.name.lower()
]

rapport = []

for fichier in sorted(shapefiles):
    print(f"\nTraitement : {fichier.name}")

    try:
        couche = gpd.read_file(fichier)

        if couche.empty:
            print("  -> Couche vide, ignorée.")
            rapport.append({
                "couche": fichier.name,
                "statut": "vide",
                "entites": 0
            })
            continue

        if couche.crs is None:
            print("  -> CRS absent, couche ignorée.")
            rapport.append({
                "couche": fichier.name,
                "statut": "CRS absent",
                "entites": 0
            })
            continue

        # Mettre la couche dans le CRS de la limite
        if couche.crs != limite.crs:
            couche = couche.to_crs(limite.crs)

        # Retirer les géométries nulles ou vides
        couche = couche[
            couche.geometry.notna() & ~couche.geometry.is_empty
        ].copy()

        if couche.empty:
            print("  -> Aucune géométrie exploitable.")
            rapport.append({
                "couche": fichier.name,
                "statut": "sans géométrie",
                "entites": 0
            })
            continue

  # Découpage spatial exact par la limite régionale
        decoupee = gpd.clip(couche, limite)

        if decoupee.empty:
            print("  -> Aucune entité dans la région.")
            rapport.append({
                "couche": fichier.name,
                "statut": "aucune entité dans la région",
                "entites": 0
            })
            continue

        # Export GeoPackage : évite les limites du format Shapefile
        sortie = OUT_DIR / f"{fichier.stem}_casablanca_settat.gpkg"
        decoupee.to_file(sortie, driver="GPKG")

        print(f"  -> {len(decoupee)} entités enregistrées")

        rapport.append({
            "couche": fichier.name,
            "statut": "OK",
            "entites": len(decoupee),
            "fichier_sortie": sortie.name
        })

    except Exception as e:
        print(f"  -> ERREUR : {e}")
        rapport.append({
            "couche": fichier.name,
            "statut": f"Erreur : {e}",
            "entites": 0
        })

# 4. Afficher le bilan
bilan = pd.DataFrame(rapport)

print("\n" + "=" * 65)
print("BILAN DU DÉCOUPAGE OSM")
print("=" * 65)
display(bilan)

print("\nNombre de couches traitées :", len(bilan))
print("Couches exportées :", (bilan["statut"] == "OK").sum())
print("Dossier de sortie :", OUT_DIR)


Dossier racine : C:\Users\espacegamers.ma\Downloads\Geodata
Dossier OSM brut : C:\Users\espacegamers.ma\Downloads\Geodata\donnees_brutes\osm
Dossier OSM traité : C:\Users\espacegamers.ma\Downloads\Geodata\donnees_traitees\casablanca_settat\osm
Limite administrative chargée.
CRS de la limite : EPSG:4326
Dossier de sortie : C:\Users\espacegamers.ma\Downloads\Geodata\donnees_traitees\casablanca_settat\osm

Traitement : gis_osm_buildings_a_free_1.shp


C:\Users\espacegamers.ma\AppData\Local\Temp\ipykernel_11268\3409736652.py:100: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  couche.geometry.notna() & ~couche.geometry.is_empty


  -> 85000 entités enregistrées

Traitement : gis_osm_landuse_a_free_1.shp


C:\Users\espacegamers.ma\AppData\Local\Temp\ipykernel_11268\3409736652.py:100: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  couche.geometry.notna() & ~couche.geometry.is_empty


  -> 11809 entités enregistrées

Traitement : gis_osm_natural_a_free_1.shp
  -> 100 entités enregistrées

Traitement : gis_osm_natural_free_1.shp
  -> 18869 entités enregistrées

Traitement : gis_osm_places_a_free_1.shp
  -> 14 entités enregistrées

Traitement : gis_osm_places_free_1.shp
  -> 404 entités enregistrées

Traitement : gis_osm_pofw_a_free_1.shp
  -> 398 entités enregistrées

Traitement : gis_osm_pofw_free_1.shp
  -> 95 entités enregistrées

Traitement : gis_osm_pois_a_free_1.shp


C:\Users\espacegamers.ma\AppData\Local\Temp\ipykernel_11268\3409736652.py:100: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  couche.geometry.notna() & ~couche.geometry.is_empty


  -> 3552 entités enregistrées

Traitement : gis_osm_pois_free_1.shp
  -> 7339 entités enregistrées

Traitement : gis_osm_protected_areas_a_free_1.shp
  -> 1 entités enregistrées

Traitement : gis_osm_railways_free_1.shp
  -> 1084 entités enregistrées

Traitement : gis_osm_roads_free_1.shp
  -> 142341 entités enregistrées

Traitement : gis_osm_traffic_a_free_1.shp
  -> 1068 entités enregistrées

Traitement : gis_osm_traffic_free_1.shp
  -> 9264 entités enregistrées

Traitement : gis_osm_transport_a_free_1.shp
  -> 64 entités enregistrées

Traitement : gis_osm_transport_free_1.shp
  -> 2697 entités enregistrées

Traitement : gis_osm_water_a_free_1.shp
  -> 544 entités enregistrées

Traitement : gis_osm_waterways_free_1.shp
  -> 506 entités enregistrées

BILAN DU DÉCOUPAGE OSM


,couche,statut,entites,fichier_sortie
0,gis_osm_buildings_a_free_1.shp,OK,85000,gis_osm_buildings_a_free_1_casablanca_settat.gpkg
1,gis_osm_landuse_a_free_1.shp,OK,11809,gis_osm_landuse_a_free_1_casablanca_settat.gpkg
2,gis_osm_natural_a_free_1.shp,OK,100,gis_osm_natural_a_free_1_casablanca_settat.gpkg
3,gis_osm_natural_free_1.shp,OK,18869,gis_osm_natural_free_1_casablanca_settat.gpkg
4,gis_osm_places_a_free_1.shp,OK,14,gis_osm_places_a_free_1_casablanca_settat.gpkg
5,gis_osm_places_free_1.shp,OK,404,gis_osm_places_free_1_casablanca_settat.gpkg
6,gis_osm_pofw_a_free_1.shp,OK,398,gis_osm_pofw_a_free_1_casablanca_settat.gpkg
7,gis_osm_pofw_free_1.shp,OK,95,gis_osm_pofw_free_1_casablanca_settat.gpkg
8,gis_osm_pois_a_free_1.shp,OK,3552,gis_osm_pois_a_free_1_casablanca_settat.gpkg
9,gis_osm_pois_free_1.shp,OK,7339,gis_osm_pois_free_1_casablanca_settat.gpkg



Nombre de couches traitées : 19
Couches exportées : 19
Dossier de sortie : C:\Users\espacegamers.ma\Downloads\Geodata\donnees_traitees\casablanca_settat\osm



## 2. Découpage des données hydrographiques CartoMaroc

Cette étape applique la même emprise administrative aux différentes
couches hydrographiques : barrages, bassins, lacs et dayas, nappes
et cours d'eau.

Les résultats sont enregistrés séparément dans
`donnees_traitees/casablanca_settat/hydrographie/`.
      

In [3]:

# ============================================================
# 5. DÉCOUPAGE DES DONNÉES HYDROGRAPHIQUES (CARTOMAROC)
# ============================================================

from pathlib import Path
import geopandas as gpd
import pandas as pd

# Dossier contenant les couches hydrographiques brutes
HYDRO_DIR = (
    ROOT / "donnees_brutes"
    / "hydro" / "extracted"
    / "hydro-maroc(CartoMaroc)"
)

# Dossier de sortie des couches hydrographiques découpées
HYDRO_OUT = (
    ROOT / "donnees_traitees"
    / "casablanca_settat"
    / "hydrographie"
)
HYDRO_OUT.mkdir(parents=True, exist_ok=True)

# Vérifications préalables
if not HYDRO_DIR.exists():
    raise FileNotFoundError(
        f"Dossier hydrographique introuvable : {HYDRO_DIR}"
    )

if "limite" not in globals():
    raise NameError(
        "La variable 'limite' est absente. "
        "Exécute d'abord la cellule de chargement de la limite régionale."
    )

# Identifier tous les shapefiles, y compris dans les sous-dossiers
hydro_files = sorted(HYDRO_DIR.rglob("*.shp"))

if not hydro_files:
    raise FileNotFoundError(
        f"Aucun shapefile trouvé dans {HYDRO_DIR}"
    )

print(f"Couches hydrographiques détectées : {len(hydro_files)}")
print(f"CRS de la limite : {limite.crs}")
print(f"Dossier de sortie : {HYDRO_OUT}")

# Tableau de suivi
rapport_hydro = []

# Découper chaque couche
for fichier in hydro_files:
    print(f"\nTraitement : {fichier.name}")

    try:
        hydro = gpd.read_file(fichier)
        nb_initial = len(hydro)

        if hydro.empty:
            print("  -> Couche vide, ignorée.")
            rapport_hydro.append({
                "couche": fichier.name,
                "statut": "Vide",
                "entites_avant": 0,
                "entites_apres": 0
            })
            continue

        if hydro.crs is None:
            print("  -> CRS absent, couche ignorée.")
            rapport_hydro.append({
                "couche": fichier.name,
                "statut": "CRS absent",
                "entites_avant": nb_initial,
                "entites_apres": 0
            })
            continue

        # Reprojection temporaire dans le CRS de la limite
        if hydro.crs != limite.crs:
            hydro = hydro.to_crs(limite.crs)

        # Retirer les géométries nulles et vides
        hydro = hydro.loc[
            hydro.geometry.notna()
            & (~hydro.geometry.is_empty)
        ].copy()

        if hydro.empty:
            print("  -> Aucune géométrie exploitable.")
            rapport_hydro.append({
                "couche": fichier.name,
                "statut": "Sans géométrie exploitable",
                "entites_avant": nb_initial,
                "entites_apres": 0
            })
            continue

        # Réparer les géométries
        hydro["geometry"] = hydro.geometry.make_valid()

        # Découpage à la limite exacte de Casablanca-Settat
        hydro_decoupee = gpd.clip(hydro, limite)

        if hydro_decoupee.empty:
            print("  -> Aucune entité dans la région.")
            rapport_hydro.append({
                "couche": fichier.name,
                "statut": "Aucune entité dans la région",
                "entites_avant": nb_initial,
                "entites_apres": 0
            })
            continue

        # Export GeoPackage
        nom_sortie = f"{fichier.stem}_casablanca_settat.gpkg"
        chemin_sortie = HYDRO_OUT / nom_sortie

        hydro_decoupee.to_file(
            chemin_sortie,
            driver="GPKG",
            index=False
        )

        print(f"  -> Entités avant : {nb_initial}")
        print(f"  -> Entités après : {len(hydro_decoupee)}")
        print(f"  -> Enregistré : {nom_sortie}")

        rapport_hydro.append({
            "couche": fichier.name,
            "statut": "OK",
            "entites_avant": nb_initial,
            "entites_apres": len(hydro_decoupee),
            "fichier_sortie": nom_sortie
        })

    except Exception as e:
        print(f"  -> ERREUR : {e}")
        rapport_hydro.append({
            "couche": fichier.name,
            "statut": f"Erreur : {e}",
            "entites_avant": None,
            "entites_apres": 0
        })

# Afficher et sauvegarder le bilan
bilan_hydro = pd.DataFrame(rapport_hydro)

display(bilan_hydro)

rapport_path = HYDRO_OUT / "rapport_decoupage_hydrographie.csv"
bilan_hydro.to_csv(
    rapport_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nDécoupage hydrographique terminé.")
print(f"Couches réussies : {(bilan_hydro['statut'] == 'OK').sum()}")
print(f"Rapport : {rapport_path}")


Couches hydrographiques détectées : 6
CRS de la limite : EPSG:4326
Dossier de sortie : C:\Users\espacegamers.ma\Downloads\Geodata\donnees_traitees\casablanca_settat\hydrographie

Traitement : barrages_(CartoMaroc).shp
  -> Entités avant : 135
  -> Entités après : 11
  -> Enregistré : barrages_(CartoMaroc)_casablanca_settat.gpkg

Traitement : Bassin-maroc(CartoMaroc).shp
  -> Entités avant : 22
  -> Entités après : 4
  -> Enregistré : Bassin-maroc(CartoMaroc)_casablanca_settat.gpkg

Traitement : Lac+Dayat(CartoMaroc).shp
  -> Entités avant : 3059
  -> Entités après : 380
  -> Enregistré : Lac+Dayat(CartoMaroc)_casablanca_settat.gpkg

Traitement : Nappes_(CartoMaroc).shp
  -> Entités avant : 67
  -> Entités après : 5
  -> Enregistré : Nappes_(CartoMaroc)_casablanca_settat.gpkg

Traitement : Oued(CartoMaroc) (1).shp
  -> Entités avant : 43837
  -> Entités après : 276
  -> Enregistré : Oued(CartoMaroc) (1)_casablanca_settat.gpkg

Traitement : Oued-Détaillé(CartoMaroc).shp
  -> Entités avan

,couche,statut,entites_avant,entites_apres,fichier_sortie
0,barrages_(CartoMaroc).shp,OK,135,11,barrages_(CartoMaroc)_casablanca_settat.gpkg
1,Bassin-maroc(CartoMaroc).shp,OK,22,4,Bassin-maroc(CartoMaroc)_casablanca_settat.gpkg
2,Lac+Dayat(CartoMaroc).shp,OK,3059,380,Lac+Dayat(CartoMaroc)_casablanca_settat.gpkg
3,Nappes_(CartoMaroc).shp,OK,67,5,Nappes_(CartoMaroc)_casablanca_settat.gpkg
4,Oued(CartoMaroc) (1).shp,OK,43837,276,Oued(CartoMaroc) (1)_casablanca_settat.gpkg
5,Oued-Détaillé(CartoMaroc).shp,OK,93681,4000,Oued-Détaillé(CartoMaroc)_casablanca_settat.gpkg



Découpage hydrographique terminé.
Couches réussies : 6
Rapport : C:\Users\espacegamers.ma\Downloads\Geodata\donnees_traitees\casablanca_settat\hydrographie\rapport_decoupage_hydrographie.csv
